# Volume 7, Chapter 8: Haar Measure, Duality and the p-adic Numbers (Python companion)

Exercises first, solutions at the end. Solving linear systems with $p$-adic numbers, sampling Haar measure, amenability seen through random walks, and exact checks of Hausdorff's free rotations and of adelic Poisson summation. Requirements: numpy.

*Before each solution, predict what the output will look like.*

## Exercises

**P8.1 (★★) CS application: solving linear systems $p$-adically.** To solve $Ax=b$ exactly with integer $A$ and $b$, Dixon's algorithm (1982) works in $\mathbb Z_p$: compute $C=A^{-1}\bmod p$ once, then find the $p$-adic digits of $x$ one at a time ($x_i=C\,r_i\bmod p$, $r_{i+1}=(r_i-Ax_i)/p$), and finally recover the rational entries of $x$ from their $p$-adic approximation by *rational reconstruction* (the extended Euclidean algorithm, stopped half-way). Implement it, check it against exact Gaussian elimination with `fractions.Fraction` for random $n\times n$ systems with entries in $[-99,99]$, and compare running times for $n=10,20,40$. Why are the intermediate numbers so much smaller?

**P8.2 (★) Haar measure is coin tossing.** Sample random elements of $\mathbb Z_p$ by drawing their digits independently and uniformly (Figure 8.2), to precision $p^{-30}$. Estimate the measure of the set of squares (Exercise 8.2) for $p=2,3,5,7$, and the local zeta integral $\int_{\mathbb Z_p}|x|_p^{s-1}\,d\mu_p$ for $p=3$, $s=2$ (Exercise 8.3). Compare with the exact values.

**P8.3 (★) Amenable and not.** Compute the Følner ratios $\frac{|gF\triangle F|}{|F|}$ for boxes in $\mathbb Z^2$ and for balls in the free group $F_2$ (Exercise 8.4). Then compute exactly, by dynamic programming, the probability $p_{2n}$ that the simple random walk returns to its start after $2n$ steps, on $\mathbb Z^2$ and on the Cayley graph of $F_2$. Kesten's theorem (1959) says that a group is amenable exactly when $p_{2n}^{1/2n}\to1$: check that $p_{2n}^{1/2n}$ tends to $1$ on $\mathbb Z^2$ and to $\frac{\sqrt3}2$ on $F_2$.

**P8.4 (★) Two checks with exact arithmetic.** (a) Verify Lemma 8.9 for all reduced words of length at most $10$ in $\phi^{\pm1},\rho^{\pm1}$: compute $w(1,0,0)=3^{-k}(a,b\sqrt2,c)$ in exact integer arithmetic and check that $3\nmid b$ whenever $w$ ends with $\phi^{\pm1}$; how many distinct points do these words produce? (b) Check the adelic Poisson formula (Theorem 8.20) numerically for $f_\infty(x)=e^{-\pi x^2}$ and the finite part $\mathbf 1_{b+M\hat{\mathbb Z}}$, with $b=\frac16$ and $M=\frac52$, by summing both sides.

## Solutions

### P8.1

In [ ]:
import numpy as np, time, random, itertools, math
from fractions import Fraction

def inv_mod_matrix(A, p):
    n = len(A)
    M = [[A[i][j] % p for j in range(n)] + [int(i == j) for j in range(n)] for i in range(n)]
    for c in range(n):
        r = next(i for i in range(c, n) if M[i][c] % p)
        M[c], M[r] = M[r], M[c]
        inv = pow(M[c][c], -1, p)
        M[c] = [v * inv % p for v in M[c]]
        for i in range(n):
            if i != c and M[i][c]:
                f = M[i][c]
                M[i] = [(vi - f * vc) % p for vi, vc in zip(M[i], M[c])]
    return [row[n:] for row in M]

def rational_reconstruction(u, m):
    """Find a/b with a = b u mod m, |a|, |b| <= sqrt(m/2), by the half extended Euclid."""
    r0, r1, s0, s1 = m, u % m, 0, 1
    bound = math.isqrt(m // 2)
    while r1 > bound:
        q = r0 // r1
        r0, r1, s0, s1 = r1, r0 - q * r1, s1, s0 - q * s1
    return Fraction(r1, s1)

def dixon(A, b, p=32003):
    n = len(A)
    C = inv_mod_matrix(A, p)
    # Hadamard bound: numerators and denominators of x are at most H = prod of row norms (times |b|)
    H = 1
    for row in A + [b]:
        H *= math.isqrt(sum(v * v for v in row)) + 1
    steps = 1
    while p ** steps < 2 * H * H:
        steps += 1
    r, x, pk = list(b), [0] * n, 1
    for _ in range(steps):
        xi = [sum(C[i][j] * r[j] for j in range(n)) % p for i in range(n)]
        Axi = [sum(A[i][j] * xi[j] for j in range(n)) for i in range(n)]
        r = [(r[i] - Axi[i]) // p for i in range(n)]       # exact division
        x = [x[i] + xi[i] * pk for i in range(n)]
        pk *= p
    return [rational_reconstruction(v, pk) for v in x], steps

def gauss_fraction(A, b):
    n = len(A)
    M = [[Fraction(v) for v in row] + [Fraction(bi)] for row, bi in zip(A, b)]
    for c in range(n):
        r = next(i for i in range(c, n) if M[i][c] != 0)
        M[c], M[r] = M[r], M[c]
        for i in range(c + 1, n):
            f = M[i][c] / M[c][c]
            M[i] = [vi - f * vc for vi, vc in zip(M[i], M[c])]
    x = [Fraction(0)] * n
    for i in reversed(range(n)):
        x[i] = (M[i][n] - sum(M[i][j] * x[j] for j in range(i + 1, n))) / M[i][i]
    return x

random.seed(1)
for n in [10, 20, 40]:
    A = [[random.randint(-99, 99) for _ in range(n)] for _ in range(n)]
    b = [random.randint(-99, 99) for _ in range(n)]
    t0 = time.perf_counter(); xd, steps = dixon(A, b); t1 = time.perf_counter()
    xg = gauss_fraction(A, b); t2 = time.perf_counter()
    print(f"n = {n}: agree = {xd == xg}, p-adic digits = {steps}, denominator has {len(str(xd[0].denominator))} digits,"
          f" Dixon {t1 - t0:.3f}s, Fraction-Gauss {t2 - t1:.3f}s")

Both methods give the same rational solution, whose common denominator is $\det A$, with about $2.3n$ decimal digits here. Fraction-based elimination drags ever larger rationals through $n^3$ operations, and every operation costs a gcd. Dixon's method computes one inverse modulo a single small prime, and then each $p$-adic digit costs a matrix–vector product with small numbers; the only big numbers are the accumulated $x$ and the final reconstruction. The time grows more slowly with $n$: for $n=40$ Dixon is already more than three times faster, and the ratio keeps growing with $n$ (the asymptotic costs are about $n^3\log n$ against $n^4$ bit operations, up to logarithms). The idea is that $\mathbb Z_p$ is a complete space in which $A^{-1}$ can be found by successive approximation, like Newton's method or Hensel's lemma (Volume 1, Chapter 12), and that a rational number with numerator and denominator below $\sqrt{p^k/2}$ is determined by its residue modulo $p^k$. Computer algebra systems (and the integer linear algebra in Sage, PARI or FLINT) are built on such $p$-adic and multi-modular methods.

### P8.2

In [ ]:
rng = np.random.default_rng(0)
K = 30

def random_zp(p, size):
    digits = rng.integers(0, p, (size, K))
    return digits

def valuation_and_unit(digits, p):
    nz = digits != 0
    v = np.where(nz.any(1), nz.argmax(1), K)
    # the unit part modulo 8 (p=2) or modulo p (p odd), read off the digits after the valuation
    idx = np.arange(len(digits))
    if p == 2:
        u = sum(digits[idx, np.minimum(v + i, K - 1)] * 2 ** i for i in range(3))
    else:
        u = digits[idx, np.minimum(v, K - 1)]
    return v, u

def is_square(digits, p):
    v, u = valuation_and_unit(digits, p)
    if p == 2:
        return (v % 2 == 0) & (u % 8 == 1)
    squares = {(x * x) % p for x in range(1, p)}
    return (v % 2 == 0) & np.isin(u, list(squares))

N = 400000
for p in [2, 3, 5, 7]:
    est = is_square(random_zp(p, N), p).mean()
    exact = 1 / 6 if p == 2 else p / (2 * (p + 1))
    print(f"p = {p}: measure of squares ~ {est:.4f}   exact {exact:.4f}")

p, s = 3, 2.0
v, _ = valuation_and_unit(random_zp(p, N), p)
print("local zeta integral:", np.mean(float(p) ** (-v * (s - 1))), " exact:", (1 - 1 / p) / (1 - p ** -s))

The estimates agree with Exercise 8.2 to about three decimals ($\frac16$ for $p=2$, $\frac38$ for $p=3$, $\frac5{12}$ for $p=5$, $\frac7{16}$ for $p=7$), and the zeta integral with $\frac{1-1/3}{1-1/9}=0.75$. Sampling the Haar measure is trivial because $\mathbb Z_p$ is, as a measure space, the space of infinite sequences of independent uniform digits: the carries in $p$-adic addition do not change the uniform distribution, which is why this product measure is translation invariant. As $p\to\infty$, the measure of the squares tends to $\frac12$: a random $p$-adic integer is almost always a unit, and half the units are squares.

### P8.3

In [ ]:
def folner_box(n):
    F = {(i, j) for i in range(n) for j in range(n)}
    G = {(i + 1, j) for (i, j) in F}
    return len(F ^ G) / len(F)

def free_ball(n):
    inv = {'a': 'A', 'A': 'a', 'b': 'B', 'B': 'b'}
    words = {''}
    frontier = ['']
    for _ in range(n):
        new = []
        for w in frontier:
            for s in 'aAbB':
                if not w or inv[s] != w[-1]:
                    new.append(w + s)
        words.update(new); frontier = new
    return words, inv

def mult(s, w, inv):
    return w[1:] if w and w[0] == inv[s] else s + w

for n in [2, 4, 6, 8]:
    B, inv = free_ball(n)
    aB = {mult('a', w, inv) for w in B}
    print(f"n = {n}: Z^2 box ratio {folner_box(2 ** n):.4f}   F_2 ball ratio {len(aB ^ B) / len(B):.4f}")

def return_probs_Z2(N):
    size = 2 * N + 1
    P = np.zeros((size, size)); P[N, N] = 1
    out = []
    for t in range(1, 2 * N + 1):
        P = 0.25 * (np.roll(P, 1, 0) + np.roll(P, -1, 0) + np.roll(P, 1, 1) + np.roll(P, -1, 1))
        if t % 2 == 0:
            out.append(P[N, N])
    return out

def return_probs_F2(N):
    # the distance from the start is a Markov chain: from 0 go to 1; from d>0 go up w.p. 3/4, down w.p. 1/4
    q = np.zeros(2 * N + 2); q[0] = 1
    out = []
    for t in range(1, 2 * N + 1):
        new = np.zeros_like(q)
        new[1] += q[0]
        new[2:] += 0.75 * q[1:-1]
        new[:-1] += 0.25 * q[1:]
        new[0] -= 0.0
        q = new
        if t % 2 == 0:
            out.append(q[0])
    return out

N = 200
pz, pf = return_probs_Z2(N), return_probs_F2(N)
for n in [10, 50, 100, 200]:
    print(f"2n = {2 * n:3d}: Z^2 p^(1/2n) = {pz[n - 1] ** (1 / (2 * n)):.4f}   F_2 p^(1/2n) = {pf[n - 1] ** (1 / (2 * n)):.4f}   (sqrt3/2 = {np.sqrt(3) / 2:.4f})")

Boxes in $\mathbb Z^2$ have Følner ratio $\frac2n$, which tends to $0$; balls in $F_2$ keep a ratio near $1$, far above the bound $\frac12$ of Exercise 8.4: in a tree, most of a ball is its boundary. The return probabilities tell the same story analytically. On $\mathbb Z^2$, $p_{2n}\approx\frac1{\pi n}$ (Volume 3, Chapter 7), which decays only polynomially, so $p_{2n}^{1/2n}\to1$, slowly. On $F_2$ the distance to the start drifts away at speed $\frac12$, and returns are exponentially rare: $p_{2n}^{1/2n}$ creeps up towards $\frac{\sqrt3}2\approx0.866$ ($0.850$ at $2n=400$, the remaining gap being a polynomial correction), the spectral radius of the walk computed by Kesten. His theorem turns amenability, a statement about invariant means that no one can write down, into a computable quantity: the norm of the averaging operator on $\ell^2(G)$.

### P8.4

In [ ]:
# (a) exact arithmetic: a vector (a, b*sqrt2, c) / 3^k is stored as integers (a, b, c, k)
def apply(s, v):
    a, b, c, k = v
    if s == 'f':  return (a - 4 * b, b + 2 * a, 3 * c, k + 1)     # phi
    if s == 'F':  return (a + 4 * b, b - 2 * a, 3 * c, k + 1)     # phi^-1
    if s == 'r':  return (3 * a, b - 2 * c, c + 4 * b, k + 1)     # rho
    if s == 'R':  return (3 * a, b + 2 * c, c - 4 * b, k + 1)     # rho^-1

inv = {'f': 'F', 'F': 'f', 'r': 'R', 'R': 'r'}
def reduced_words(L):
    words = ['']
    for _ in range(L):
        words = [w + s for w in words for s in 'fFrR' if not w or inv[s] != w[-1]]
        yield from words

ok, points, count = True, set(), 0
for w in reduced_words(10):
    v = (1, 0, 0, 0)
    for s in reversed(w):          # the rightmost letter acts first
        v = apply(s, v)
    count += 1
    if w[-1] in 'fF' and v[1] % 3 == 0:
        ok = False
    a, b, c, k = v
    g = math.gcd(math.gcd(a, b), math.gcd(c, 3 ** k))
    points.add((a // g, b // g, c // g, 3 ** k // g))
print("words checked:", count, " 3 never divides b:", ok, " distinct image points:", len(points))

# (b) adelic Poisson: sum f_inf(b + M n) = (1/M) sum_k hat f_inf(k/M) e(-b k / M), with hat f_inf = f_inf for the Gaussian
b, M = Fraction(1, 6), Fraction(5, 2)
lhs = sum(np.exp(-np.pi * float(b + M * n) ** 2) for n in range(-50, 51))
rhs = sum(np.exp(-np.pi * float(Fraction(k) / M) ** 2) * np.exp(-2j * np.pi * float(b * k / M)) for k in range(-50, 51)) / float(M)
print("left side", lhs, "  right side", rhs)

All $118\,096$ non-empty reduced words of length at most $10$ pass the test of the lemma. They produce exactly $59\,049=3^{10}$ distinct points, and the count explains itself: $\rho$ fixes the $x$-axis, so $w\rho^{\pm1}$ sends $(1,0,0)$ where $w$ does, and the distinct images are those of the empty word and of the $2\cdot3^{k-1}$ words of each length $k\le10$ ending with $\phi^{\pm1}$, that is $1+\sum_{k=1}^{10}2\cdot3^{k-1}=3^{10}$. These are all different, as the lemma predicts: the orbit of a point under the free group is a tree drawn on the sphere. The two sides of the adelic Poisson formula agree to machine precision; the right side is real although its terms are not, because the terms for $k$ and $-k$ are conjugate. With $b=\frac16$, $M=\frac52$ the finite part is the indicator of $\{x:x_2\in\frac16+\frac12\mathbb Z_2,\ x_3\in\frac16+\mathbb Z_3,\ x_5\in\frac16+5\mathbb Z_5,\ x_p\in\mathbb Z_p\text{ otherwise}\}$, and the identity mixes the real Gaussian with the arithmetic of $2$, $3$ and $5$.